# ELMo 情感分析
使用 AllenNLP ELMo 预训练模型对 `sentiment_dataset_1000.csv` 做二分类情感识别。

**标签：** 0 = Positive，1 = Negative

**安装依赖（首次运行）：**
```
pip install allennlp torch pandas scikit-learn
```

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report
from allennlp.modules.elmo import Elmo, batch_to_ids

# ELMo 本地预训练权重（已下载到当前目录）
OPTIONS_FILE = "elmo_options.json"
WEIGHT_FILE  = "elmo_weights.hdf5"
ELMO_DIM = 256  # ELMo 输出维度

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


## 1. 加载数据

In [10]:
df = pd.read_csv('sentiment_dataset_1000.csv')
print(f'数据量: {len(df)} 条')
print(f'标签分布:\n{df["label"].value_counts()}')
df.head()

数据量: 1000 条
标签分布:
label
0    500
1    500
Name: count, dtype: int64


,text,label
0,Life is good.,0
1,"I missed the bus, what a bad day.",1
2,Feeling grateful for everything.,0
3,I had an amazing day!,0
4,I can't stop smiling today.,0


## 2. Dataset & DataLoader

In [11]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = [t.split() for t in texts]  # 空格分词
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]


def collate_fn(batch):
    texts, labels = zip(*batch)
    char_ids = batch_to_ids(list(texts))  # ELMo 字符级 ID
    return char_ids, torch.tensor(labels, dtype=torch.long)


texts = df['text'].tolist()
labels = df['label'].tolist()

tr_texts, val_texts, tr_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

tr_loader  = DataLoader(SentimentDataset(tr_texts, tr_labels),
                        batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(SentimentDataset(val_texts, val_labels),
                        batch_size=16, collate_fn=collate_fn)

print(f'训练集: {len(tr_texts)} 条  验证集: {len(val_texts)} 条')

训练集: 800 条  验证集: 200 条


## 3. ELMo 分类模型

In [12]:
class ELMoClassifier(nn.Module):
    """ELMo 嵌入 + 均值池化 + MLP 分类器"""

    def __init__(self, elmo_dim=ELMO_DIM, hidden_dim=128,
                 num_classes=2, dropout=0.3):
        super().__init__()
        self.elmo = Elmo(OPTIONS_FILE, WEIGHT_FILE,
                         num_output_representations=1, dropout=0)
        self.classifier = nn.Sequential(
            nn.Linear(elmo_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, char_ids):
        out = self.elmo(char_ids)
        reps = out['elmo_representations'][0]       # (B, T, D)
        mask = out['mask'].float()                  # (B, T)
        lengths = mask.sum(dim=1, keepdim=True).clamp(min=1)
        pooled = (reps * mask.unsqueeze(-1)).sum(dim=1) / lengths
        return self.classifier(pooled)


model = ELMoClassifier().to(device)
print(model)

ELMoClassifier(
  (elmo): Elmo(
    (_elmo_lstm): _ElmoBiLm(
      (_token_embedder): _ElmoCharacterEncoder(
        (char_conv_0): Conv1d(16, 32, kernel_size=(1,), stride=(1,))
        (char_conv_1): Conv1d(16, 32, kernel_size=(2,), stride=(1,))
        (char_conv_2): Conv1d(16, 64, kernel_size=(3,), stride=(1,))
        (char_conv_3): Conv1d(16, 128, kernel_size=(4,), stride=(1,))
        (char_conv_4): Conv1d(16, 256, kernel_size=(5,), stride=(1,))
        (char_conv_5): Conv1d(16, 512, kernel_size=(6,), stride=(1,))
        (char_conv_6): Conv1d(16, 1024, kernel_size=(7,), stride=(1,))
        (_highways): Highway(
          (_layers): ModuleList(
            (0): Linear(in_features=2048, out_features=4096, bias=True)
          )
        )
        (_projection): Linear(in_features=2048, out_features=128, bias=True)
      )
      (_elmo_lstm): ElmoLstm(
        (forward_layer_0): LstmCellWithProjection(
          (input_linearity): Linear(in_features=128, out_features=4096, bias=Fal

## 4. 训练

In [13]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

def train_epoch(loader):
    model.train()
    total_loss = 0
    for char_ids, lbls in loader:
        char_ids, lbls = char_ids.to(device), lbls.to(device)
        optimizer.zero_grad()
        loss = criterion(model(char_ids), lbls)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for char_ids, lbls in loader:
            logits = model(char_ids.to(device))
            preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
            trues.extend(lbls.tolist())
    return preds, trues


epochs = 5
for epoch in range(1, epochs + 1):
    loss = train_epoch(tr_loader)
    preds, trues = evaluate(val_loader)
    acc = accuracy_score(trues, preds)
    f1  = f1_score(trues, preds, average='weighted')
    print(f'Epoch {epoch}/{epochs}  Loss: {loss:.4f}  Acc: {acc:.4f}  F1: {f1:.4f}')

Epoch 1/5  Loss: 0.4560  Acc: 0.9200  F1: 0.9200
Epoch 2/5  Loss: 0.1152  Acc: 1.0000  F1: 1.0000
Epoch 3/5  Loss: 0.0293  Acc: 1.0000  F1: 1.0000
Epoch 4/5  Loss: 0.0104  Acc: 1.0000  F1: 1.0000
Epoch 5/5  Loss: 0.0057  Acc: 1.0000  F1: 1.0000


## 5. 评估报告

In [14]:
preds, trues = evaluate(val_loader)
print(classification_report(trues, preds, target_names=['Positive', 'Negative']))

              precision    recall  f1-score   support

    Positive       1.00      1.00      1.00       100
    Negative       1.00      1.00      1.00       100

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



## 6. 示例预测

In [15]:
label_map = {0: 'Positive', 1: 'Negative'}

def predict(texts):
    model.eval()
    char_ids = batch_to_ids([t.split() for t in texts]).to(device)
    with torch.no_grad():
        return torch.argmax(model(char_ids), dim=1).cpu().tolist()


samples = [
    "I love this product!",
    "This is terrible, I hate it.",
    "Feeling great today!",
    "I am so disappointed with the results.",
]

for text, pred in zip(samples, predict(samples)):
    print(f'[{label_map[pred]}]  {text}')

[Positive]  I love this product!
[Negative]  This is terrible, I hate it.
[Positive]  Feeling great today!
[Negative]  I am so disappointed with the results.


## 7. 保存模型

In [16]:
torch.save(model.state_dict(), 'elmo_sentiment_model.pth')
print('模型已保存: elmo_sentiment_model.pth')

模型已保存: elmo_sentiment_model.pth
